# A/B Тест: Qwen3-VL-8B vs Qwen3-VL-2B

Сравниваем два размера модели для VQA-аннотаций мемов.
- **8B** — медленнее, потенциально точнее
- **2B** — быстрее, потенциально менее детальна

Цель: обосновать выбор модели для полного прогона на 10k+ мемов.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
# === Загрузка данных ===
def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    continue
    return records

data_8b = load_jsonl('../data/processed/comparison_8b.jsonl')
data_2b = load_jsonl('../data/processed/comparison_2b.jsonl')

print(f'8B results: {len(data_8b)}')
print(f'2B results: {len(data_2b)}')

In [ ]:
df_8b = pd.DataFrame(data_8b)
df_2b = pd.DataFrame(data_2b)

df_8b['model'] = '8B'
df_2b['model'] = '2B'

print('8B columns:', df_8b.columns.tolist())
print('2B columns:', df_2b.columns.tolist())

## 1. JSON Parse Success Rate
Сколько раз модель вернула валидный JSON?

In [ ]:
success_8b = df_8b['parse_success'].sum()
success_2b = df_2b['parse_success'].sum()

total_8b = len(df_8b)
total_2b = len(df_2b)

print(f'8B: {success_8b}/{total_8b} ({success_8b/total_8b*100:.1f}%) successful JSON')
print(f'2B: {success_2b}/{total_2b} ({success_2b/total_2b*100:.1f}%) successful JSON')

fig, ax = plt.subplots(figsize=(8, 5))
models = ['8B', '2B']
rates = [success_8b/total_8b*100, success_2b/total_2b*100]
colors = ['#e74c3c', '#2ecc71']
bars = ax.bar(models, rates, color=colors, edgecolor='black')
ax.set_ylabel('Success Rate (%)')
ax.set_title('JSON Parse Success Rate')
ax.set_ylim(0, 105)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{rate:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Скорость (время на картинку)

In [ ]:
avg_8b_time = df_8b['time_seconds'].mean()
avg_2b_time = df_2b['time_seconds'].mean()

print(f'8B avg time: {avg_8b_time:.2f}s/image')
print(f'2B avg time: {avg_2b_time:.2f}s/image')

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(['8B', '2B'], [avg_8b_time, avg_2b_time], color=['#e74c3c', '#2ecc71'], edgecolor='black')
ax.set_ylabel('Seconds per image')
ax.set_title('Processing Speed')
for i, v in enumerate([avg_8b_time, avg_2b_time]):
    ax.text(i, v + 0.2, f'{v:.1f}s', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

n_images = 10000
eta_8b = n_images * avg_8b_time / 3600
eta_2b = n_images * avg_2b_time / 3600
print(f'\nETA for {n_images} images:')
print(f'  8B: {eta_8b:.1f} hours')
print(f'  2B: {eta_2b:.1f} hours')

## 3. Длина описаний (Caption Length)

In [ ]:
captions_8b = df_8b[df_8b['parse_success']]['caption'].dropna().astype(str)
captions_2b = df_2b[df_2b['parse_success']]['caption'].dropna().astype(str)

len_8b = captions_8b.apply(lambda x: len(x.split()))
len_2b = captions_2b.apply(lambda x: len(x.split()))

if len(len_8b) > 0:
    print(f'8B caption length: mean={len_8b.mean():.1f} words, median={len_8b.median():.0f}')
if len(len_2b) > 0:
    print(f'2B caption length: mean={len_2b.mean():.1f} words, median={len_2b.median():.0f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if len(len_8b) > 0:
    axes[0].hist(len_8b, bins=20, alpha=0.7, color='#e74c3c')
axes[0].set_title('8B Caption Length (words)')
axes[0].set_xlabel('Words')
if len(len_2b) > 0:
    axes[1].hist(len_2b, bins=20, alpha=0.7, color='#2ecc71')
axes[1].set_title('2B Caption Length (words)')
axes[1].set_xlabel('Words')
plt.tight_layout()
plt.show()

## 4. Количество объектов

In [ ]:
def get_obj_count(df):
    counts = []
    for _, row in df[df['parse_success']].iterrows():
        objs = row.get('objects', [])
        if isinstance(objs, list):
            counts.append(len(objs))
        elif isinstance(objs, str):
            counts.append(len(objs.split(',')))
        else:
            counts.append(0)
    return counts

obj_8b = get_obj_count(df_8b)
obj_2b = get_obj_count(df_2b)

if obj_8b:
    print(f'8B objects per image: mean={np.mean(obj_8b):.1f}')
if obj_2b:
    print(f'2B objects per image: mean={np.mean(obj_2b):.1f}')

## 5. Распределение тонов

In [ ]:
def get_tones(df):
    tones = df[df['parse_success']]['tone'].dropna().tolist()
    return Counter(tones)

tones_8b = get_tones(df_8b)
tones_2b = get_tones(df_2b)

print('8B tones:', dict(tones_8b))
print('2B tones:', dict(tones_2b))

if tones_8b or tones_2b:
    all_tones = sorted(set(list(tones_8b.keys()) + list(tones_2b.keys())))
    x = np.arange(len(all_tones))
    w = 0.35
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - w/2, [tones_8b.get(t, 0) for t in all_tones], w, label='8B', color='#e74c3c')
    ax.bar(x + w/2, [tones_2b.get(t, 0) for t in all_tones], w, label='2B', color='#2ecc71')
    ax.set_xticks(x)
    ax.set_xticklabels(all_tones, rotation=45)
    ax.set_title('Tone Distribution')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Примеры: Side-by-Side

In [ ]:
merged = pd.merge(
    df_8b[df_8b['parse_success']][['filename', 'caption', 'objects', 'tone', 'main_idea']],
    df_2b[df_2b['parse_success']][['filename', 'caption', 'objects', 'tone', 'main_idea']],
    on='filename',
    suffixes=('_8b', '_2b'),
    how='inner'
)

print(f'Images with BOTH models successful: {len(merged)}')

pd.set_option('display.max_colwidth', 80)
if len(merged) > 0:
    display(merged[['filename', 'caption_8b', 'caption_2b', 'tone_8b', 'tone_2b']].head(10))

## 7. Agreement Score

In [ ]:
if len(merged) > 0:
    tone_agree = (merged['tone_8b'] == merged['tone_2b']).mean()
    print(f'Tone agreement: {tone_agree*100:.1f}%')
    
    overlaps = []
    for _, row in merged.iterrows():
        words_8b = set(str(row['caption_8b']).lower().split())
        words_2b = set(str(row['caption_2b']).lower().split())
        if words_8b and words_2b:
            overlap = len(words_8b & words_2b) / len(words_8b | words_2b)
            overlaps.append(overlap)
    
    if overlaps:
        print(f'Caption word overlap (Jaccard): {np.mean(overlaps)*100:.1f}%')
else:
    print('No overlapping successful results to compare.')

## 8. Итоговая таблица

In [ ]:
summary = {
    'Metric': ['JSON Success Rate', 'Avg Time (s/img)', 'ETA for 10k (hours)',
               'Avg Caption Length (words)', 'Avg Objects Count'],
    '8B': [
        f'{success_8b/total_8b*100:.1f}%',
        f'{avg_8b_time:.1f}',
        f'{10000*avg_8b_time/3600:.1f}',
        f'{len_8b.mean():.1f}' if len(len_8b) > 0 else 'N/A',
        f'{np.mean(obj_8b):.1f}' if obj_8b else 'N/A'
    ],
    '2B': [
        f'{success_2b/total_2b*100:.1f}%',
        f'{avg_2b_time:.1f}',
        f'{10000*avg_2b_time/3600:.1f}',
        f'{len_2b.mean():.1f}' if len(len_2b) > 0 else 'N/A',
        f'{np.mean(obj_2b):.1f}' if obj_2b else 'N/A'
    ]
}

summary_df = pd.DataFrame(summary)
display(summary_df)

print()
print('Рекомендация: выбираем модель с лучшим соотношением success rate / скорость.')